# **INSTALLATION DES BIBLIOTHEQUES**

- NLTK : (Tokenisation,stop word,lemmantisation)
- SPACY :(plus moderne avec comprehension du français)
- PANDAS : traitement des données
- MATPLOTLIB + SEABORN : Visualisation des données
- SCIKIT_LEARN : ML , vectorisation
- WordCloud    : génération de nuages de mots

# Installation des librairies

- création d'un fichier requirements.txt 
- ensuite pip install --default-timeout=1000 -r requirements.txt

# **Imports principaux**

In [1]:
import requests
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

import nltk
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Téléchargement des ressources NLTK

In [2]:
import nltk

nltk.download('punkt')       # Tokeniseur de phrases et mots
nltk.download('punkt_tab')   # Ressource complémentaire (versions récentes de NLTK)
nltk.download('stopwords')   # Listes de mots vides (français, anglais...)
nltk.download('wordnet')     # Base lexicale pour la lemmatisation
nltk.download('omw-1.4')    # Open Multilingual Wordnet (multilingue)



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lklk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lklk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lklk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lklk\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\lklk\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
# Vérification dans Python :
import spacy
nlp = spacy.load('fr_core_news_sm')
print('Modèle SpaCy chargé avec succès !')


Modèle SpaCy chargé avec succès !


## ✏️  Travail demandé

### b)  Affichez la version de chaque librairie installée :


In [5]:
import nltk, spacy, sklearn, pandas
print(nltk.__version__, spacy.__version__, sklearn.__version__, pandas.__version__)

3.9.4 3.8.14 1.9.0 3.0.3


### c)  Vérifiez que le modèle SpaCy est chargé en analysant la phrase :


In [6]:
doc = nlp('Edmond Dantès fut injustement emprisonné au château d\'If.')
print([(t.text, t.pos_) for t in doc])

[('Edmond', 'PROPN'), ('Dantès', 'PROPN'), ('fut', 'AUX'), ('injustement', 'ADV'), ('emprisonné', 'VERB'), ('au', 'ADP'), ('château', 'NOUN'), ("d'", 'ADP'), ('If', 'PROPN'), ('.', 'PUNCT')]


# Chargement du corpus

In [7]:
import requests

URL = 'https://www.gutenberg.org/files/17989/17989-0.txt'

# Téléchargement
response = requests.get(URL)
response.encoding = 'utf-8'
raw_text = response.text

print(f'Téléchargement terminé.')
print(f'Taille brute : {len(raw_text):,} caractères')
print(f'Aperçu :\n{raw_text[:300]}')



Téléchargement terminé.
Taille brute : 727,600 caractères
Aperçu :
*** START OF THE PROJECT GUTENBERG EBOOK 17989 ***




LE COMTE DE MONTE-CRISTO

Alexandre Dumas

Tome I (1845-1846)




Table des matières


Alexandre Dumas
I  Marseille.--L'arrivée.
II  Le père et le fils.
III  Les Catalans.
IV  Complot.
V  Le repas des fiançailles.
VI  Le substitut du procureur d


## Extraction du contenu (sans les métadonnées Gutenberg)

In [8]:
# Le fichier Gutenberg contient un en-tête et un pied de page
# On extrait uniquement le roman
DEBUT = '*** START OF THE PROJECT GUTENBERG EBOOK 17989 ***'
FIN   = '*** END OF THE PROJECT GUTENBERG EBOOK 17989 ***'

roman = raw_text.split(DEBUT)[1].split(FIN)[0]

print(f'Taille du roman (sans métadonnées) : {len(roman):,} caractères')
print(f'Nombre de lignes : {roman.count(chr(10)):,}')
print(f'Aperçu :\n{roman[:500]}')


Taille du roman (sans métadonnées) : 727,501 caractères
Nombre de lignes : 16,694
Aperçu :





LE COMTE DE MONTE-CRISTO

Alexandre Dumas

Tome I (1845-1846)




Table des matières


Alexandre Dumas
I  Marseille.--L'arrivée.
II  Le père et le fils.
III  Les Catalans.
IV  Complot.
V  Le repas des fiançailles.
VI  Le substitut du procureur du roi.
VII  L'interrogatoire.
VIII  Le château d'If.
IX  Le soir des fiançailles.
X  Le petit cabinet des Tuileries.
XI  L'Ogre de Corse.
XII  Le père et le fils.
XIII  Les Cent-Jours.
XIV  Le prisonnier furieux et le prisonnier fou.
XV  Le numéro 34


## Sauvegarde locale (optionnel)

In [9]:
# Sauvegarder le roman en local pour éviter de re-télécharger
with open('monte_cristo.txt', 'w', encoding='utf-8') as f:
    f.write(roman)

# Chargement depuis le fichier local :
# with open('monte_cristo.txt', encoding='utf-8') as f:
#     roman = f.read()


## ✏️  **Travail demandé**

a)  Téléchargez le roman et affichez :
- Le nombre total de caractères (len)
    

In [12]:
total_caractere = len(roman)
print(f"Nombre total de caractère {total_caractere:,}")

Nombre total de caractère 727,501


- Le nombre total de lignes (count('\n'))
  

In [13]:
NBR_ligne =  roman.count('\n')
print(f"Le nombre total de ligne: {NBR_ligne:,}")

Le nombre total de ligne: 16,694


  - Les 500 premiers caractères


In [14]:
caractere_500 = roman[:500]
print(f"Les 500 premières lignes : \n{caractere_500}")

Les 500 premières lignes : 





LE COMTE DE MONTE-CRISTO

Alexandre Dumas

Tome I (1845-1846)




Table des matières


Alexandre Dumas
I  Marseille.--L'arrivée.
II  Le père et le fils.
III  Les Catalans.
IV  Complot.
V  Le repas des fiançailles.
VI  Le substitut du procureur du roi.
VII  L'interrogatoire.
VIII  Le château d'If.
IX  Le soir des fiançailles.
X  Le petit cabinet des Tuileries.
XI  L'Ogre de Corse.
XII  Le père et le fils.
XIII  Les Cent-Jours.
XIV  Le prisonnier furieux et le prisonnier fou.
XV  Le numéro 34


- Les 200 derniers caractères (pour voir la fin du fichier Gutenberg)

In [15]:
caractere_200 = roman[-200:]
print(f"Les 200 dernières lignes : \n{caractere_200}")

Les 200 dernières lignes : 
donna sans réserve et finit
par retomber haletant, brûlé de fatigue, épuisé de volupté, sous les
baisers de ces maîtresses de marbre et sous les enchantements de ce rêve
inouï.

FIN DU TOME PREMIER





b)  Comptez le nombre de mots en utilisant simplement .split() (estimation rapide, avant toute tokenisation avancée)


In [18]:
mots = roman.split()
NBR_mot = len(mots)

print(f"Le nombre total de mot : {NBR_mot:,}")

Le nombre total de mot : 121,579


c)  Combien de fois apparaît le nom 'Dantès' dans le roman ? Utilisez roman.lower().count('dantès')


In [21]:
NBR_Dantes = roman.lower().count('dantès')
print(f"Le nombre de fois qu'apparait le mot Dantès : {NBR_Dantes} fois")

Le nombre de fois qu'apparait le mot Dantès : 777 fois


d)  Cherchez les 5 premières lignes contenant le mot 'vengeance'.

In [28]:
lignes = roman.split('\n')
ligne_vengeance = [mot for mot in lignes if 'vengeance' in mot.lower()]
print(f'Le nombre de ligne ou apparaisse le mot vengeance : {len(ligne_vengeance)} ')


for i,phrase in enumerate(ligne_vengeance[:5],1):
    print("")
    print(f" {i} , {phrase.split()}")

Le nombre de ligne ou apparaisse le mot vengeance : 12 

 1 , ['que', 'Fernand', 'surtout', 'était', 'terrible', 'dans', 'sa', 'vengeance.»']

 2 , ['«À', 'la', 'bonne', 'heure,', 'continua', 'Danglars;', 'ainsi', 'votre', 'vengeance', 'aurait', 'le']

 3 , ['menaçant', 'et', 'fort', 'pour', 'toutes', 'les', 'vengeances;', 'alors', 'il', 'manifesta', 'à', 'M.']

 4 , ['qui,', 'pour', 'lui', 'aussi,', 'était', 'devenu', 'messager', "d'une", 'rude', 'vengeance.', 'Alors,']

 5 , ['Il', 'se', 'disait', 'que', "c'était", 'la', 'haine', 'des', 'hommes', 'et', 'non', 'la', 'vengeance', 'de', 'Dieu']


e)  Estimez le nombre de pages du roman (supposer 300 mots / page).

In [37]:
hypotese_mot_par_page = 300
estimation_nbr_page = NBR_mot / hypotese_mot_par_page

print(f"Le nombre de mot : {NBR_mot:,}")
print(f"Hypothèse de mot par page :{hypotese_mot_par_page}")
print(f"Estimation du nombre de page : {estimation_nbr_page:.0f} Pages")

Le nombre de mot : 121,579
Hypothèse de mot par page :300
Estimation du nombre de page : 405 Pages


### *Questions de réflexion*

*Q1.  Pourquoi est-il important d'extraire uniquement le contenu du roman et de supprimer les métadonnées Gutenberg avant toute analyse ?*

- dans ce contexte les metadonnées, ne font pas parties du contenu litteraire

*Q2.  Quelle est la différence entre compter les mots avec .split() et utiliser word_tokenize ? Quand préférer l'un ou l'autre ?*


In [44]:
texte_exemple = "C'est l'histoire d'Edmond Dantès. Il crie : 'Vengeance!' et s'échappe."

decoupe_w_split = texte_exemple.split()
print(f"La découpe avec split :\n{decoupe_w_split} ")



from nltk.tokenize import word_tokenize

decoupe_w_wt = word_tokenize(texte_exemple)
print()
print(f"La découpe avec word_tokenize :\n{decoupe_w_wt} ")


La découpe avec split :
["C'est", "l'histoire", "d'Edmond", 'Dantès.', 'Il', 'crie', ':', "'Vengeance!'", 'et', "s'échappe."] 

La découpe avec word_tokenize :
["C'est", "l'histoire", "d'Edmond", 'Dantès', '.', 'Il', 'crie', ':', "'Vengeance", '!', "'", 'et', "s'échappe", '.'] 


- La découpe avec le split n'est pas très précise , y'a des sauts de certaines ponctuation , elle est plus utilisée pour une analyse rapide
- LA découpe avec word_tokenize est plus précise , elle peut etre utilisée pour la lemmantisation

*Q3.  La fréquence d'un nom de personnage est-elle un bon indicateur de son importance dans le roman ? Justifiez.*

In [87]:
from chargement.load import  LoadRoman

roman = LoadRoman.Load('https://www.gutenberg.org/files/17989/17989-0.txt')  
print(roman)






LE COMTE DE MONTE-CRISTO

Alexandre Dumas

Tome I (1845-1846)




Table des matières


Alexandre Dumas
I  Marseille.--L'arrivée.
II  Le père et le fils.
III  Les Catalans.
IV  Complot.
V  Le repas des fiançailles.
VI  Le substitut du procureur du roi.
VII  L'interrogatoire.
VIII  Le château d'If.
IX  Le soir des fiançailles.
X  Le petit cabinet des Tuileries.
XI  L'Ogre de Corse.
XII  Le père et le fils.
XIII  Les Cent-Jours.
XIV  Le prisonnier furieux et le prisonnier fou.
XV  Le numéro 34 et le numéro 27.
XVI  Un savant italien.
XVII  La chambre de l'abbé.
XVIII  Le trésor.
XIX  Le troisième accès.
XX  Le cimetière du château d'If.
XXI  L'île de Tiboulen.
XXII  Les contrebandiers.
XXIII  L'île de Monte-Cristo.
XXIV  Éblouissement.
XXV  L'inconnu.
XXVI  L'auberge du pont du Gard.
XXVII  Le récit.
XXVIII  Les registres des prisons.
XXIX  La maison Morrel.
XXX  Le cinq septembre.
XXXI  Italie.--Simbad le marin.




I

Marseille.--L'arrivée.


Le 24 février 1815, la vigie de Notre-D

# 🧹Partie 3	Nettoyage du texte


In [45]:
# Observez le texte brut avant nettoyage
print(repr(roman[1000:1200]))   # repr() montre les caractères cachés (\n, \t...)


"ame de la Garde signala le\ntrois-mâts le _Pharaon_, venant de Smyrne, Trieste et Naples.\n\nComme d'habitude, un pilote côtier partit aussitôt du port, rasa le\nchâteau d'If, et alla aborder le navire en"


### Fonction de nettoyage

In [ ]:
import re

def clean_text(text):
    """Nettoie un texte pour une analyse NLP en français."""

    # 1. Mise en minuscules
    text = text.lower()

    # 2. Suppression des marqueurs d'italique Gutenberg (_mot_)
    text = re.sub(r'_([^_]+)_', r'\1', text)

    # 3. Remplacement des sauts de ligne et tabulations par des espaces
    text = re.sub(r'[\n\r\t]+', ' ', text)

    text = re.sub()
    # 4. Suppression de la ponctuation et des caractères spéciaux

    #    On conserve les lettres françaises accentuées
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ' ]", ' ', text)

    # 5. Normalisation des apostrophes typographiques
    text = re.sub(r"[''`]", "'", text)

    # 6. Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text)

    # 7. Suppression des espaces en début et fin
    text = text.strip()
    
    return text



## ✏️  Travail demandé

### a)  Appliquez clean_text sur le roman. Affichez la taille avant et après.


In [55]:
taille_avant_nettoyage= len(roman)


text_clean = clean_text(roman)
taille_apres_nettoyage= len(text_clean)

print(f"Taille avant le nettoyage:\n{taille_avant_nettoyage:,}\n")
print(f"Taille après le nettoyage:\n{taille_apres_nettoyage:,}")





Taille avant le nettoyage:
727,501

Taille après le nettoyage:
695,422


### b)  Testez votre fonction sur cet extrait bruité :

In [50]:
text_brute = '«Ah! c\'est vous, Dantès!» cria l\'homme... (numéro 34)\n\n-- Il est mort!!'

resultat_text_brute = clean_text(text_brute)
print(resultat_text_brute)

ah c'est vous dantès cria l'homme numéro il est mort


- Suppression de saut de ligne
- suppression de ponctuation
- suppression de chiffres
- suppression de guillemets


### c)  Le nom 'Monte-Cristo' contient un tiret. Après nettoyage, devient-il 'monte cristo' (2 mots) ou 'montécristo' ? Proposez une solution pour le conserver en un seul token.
   


### - Avant modification de la fonction

In [54]:
test ='Monte-Cristo'
test_resultat = clean_text(test)
print(test_resultat.split())
print("Le resultat est affiché en deux mots, ce qu'on ne veut pas")

['monte', 'cristo']
Le resultat est affiché en deux mots, ce qu'on ne veut pas


In [ ]:
import re

def clean_text_v1(text):
    """Nettoie un texte pour une analyse NLP en français."""

    # 1. Mise en minuscules
    text = text.lower()

    # 2. Suppression des marqueurs d'italique Gutenberg (_mot_)
    text = re.sub(r'_([^_]+)_', r'\1', text)

    # 3. Remplacement des sauts de ligne et tabulations par des espaces
    text = re.sub(r'[\n\r\t]+', ' ', text)

    text = re.sub(r'-','_',text) # ===> Je remplace d'abord le trait d'union par tiret du 8

    # 4. Suppression de la ponctuation et des caractères spéciaux
    #    On conserve les lettres françaises accentuées
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ'_ ]", ' ', text)

    # 5. Normalisation des apostrophes typographiques
    text = re.sub(r"[''`]", "'", text)

    # 6. Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text)

    # 7. Suppression des espaces en début et fin
    text = text.strip()
    
    return text



### - Après modification de la fonction

In [64]:
test ='Monte-Cristo'
test_resultat = clean_text_v1(test)
print(test_resultat.split())
print("Affichage en un seul token")

['monte_cristo']
Affichage en un seul token


### d)  Modifiez clean_text pour également supprimer les nombres isolés (ex: '24', '1815').

In [65]:
import re

def clean_text_v2(text):
    """Nettoie un texte pour une analyse NLP en français."""

    # 1. Mise en minuscules
    text = text.lower()

    # 2. Suppression des marqueurs d'italique Gutenberg (_mot_)
    text = re.sub(r'_([^_]+)_', r'\1', text)

    # 3. Remplacement des sauts de ligne et tabulations par des espaces
    text = re.sub(r'[\n\r\t]+', ' ', text)

    text = re.sub(r'-','_',text) # ===> Je remplace d'abord le trait d'union par tiret du 8

    # 4. Suppression de la ponctuation et des caractères spéciaux
    #    On conserve les lettres françaises accentuées
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ'_ ]", ' ', text)

    text = re.sub(r'\b\d+\b', ' ', text) # ===> supprimer les nombres isolés


    # 5. Normalisation des apostrophes typographiques
    text = re.sub(r"[''`]", "'", text)

    # 6. Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text)

    # 7. Suppression des espaces en début et fin
    text = text.strip()
    
    return text



In [66]:
test = "Bonjour c'est un test47 des  nombres67 is0lés4"

resultat_test = clean_text_v2(test)
print(resultat_test)

bonjour c'est un test des nombres is lés


### e)  Écrivez une version clean_text_leger qui ne fait que la mise en minuscule et la suppression des espaces multiples. Comparez les résultats.

In [68]:
import re

def clean_text_v3(text):
    """Nettoie un texte pour une analyse NLP en français."""

    # 1. Mise en minuscules
    text = text.lower()

    # 2. Suppression des marqueurs d'italique Gutenberg (_mot_)
    text = re.sub(r'_([^_]+)_', r'\1', text)

    # 3. Remplacement des sauts de ligne et tabulations par des espaces
    text = re.sub(r'[\n\r\t]+', ' ', text)

    text = re.sub(r'-','_',text) # ===> Je remplace d'abord le trait d'union par tiret du 8

    # 4. Suppression de la ponctuation et des caractères spéciaux
    #    On conserve les lettres françaises accentuées
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ'_ ]", ' ', text)

    text = re.sub(r'\b\d+\b', ' ', text) # ===> supprimer les nombres isolés

    # 5. Normalisation des apostrophes typographiques
    text = re.sub(r"[''`]", "'", text)

    # 6. Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text)

    # 7. Suppression des espaces en début et fin
    text = text.strip()
    
    return text



In [69]:
roman_clean = clean_text_v3(roman)

## ❓  Questions de réflexion

### Q1.  Pourquoi faut-il nettoyer un texte avant de l'analyser en NLP ? Citez 3 raisons.

- Enlever les metadonnées nont nécessaires
- on doit nettoyer parce que le texte doit etre normaliser pour eviter les doublons ou des totologies
- 


### Q2.  Quels problèmes peuvent apparaître si on n'effectue pas de nettoyage ? (au moins 3 exemples concrets sur ce roman)

- de nombreuse ponctuations non nécessaires pour le model 
- une mauvaise lemmantisation sur les noms composés
- les espaces multiples 

### Q3.  Le nettoyage supprime-t-il de l'information utile ? Dans quel contexte la ponctuation pourrait-elle être informative ?

- le nettoyage ne supprimer pas forcemnt l'information dans le cas ou la suppression des stop word n'entraine pas une incomprehension du model
- 

### Q4.  Faut-il toujours appliquer toutes les étapes de nettoyage ? Donnez un exemple où une étape serait contre-productive.

# ✂️ Partie 4	Tokenisation


## a)  Tokenisez le roman complet avec NLTK. Affichez les 50 premiers tokens.

In [83]:
from nltk.tokenize import word_tokenize

tokens = word_tokenize(roman_clean)
print(f'\n50 premiers tokens :')
print(tokens[:50])


50 premiers tokens :
['le', 'comte', 'de', 'monte_cristo', 'alexandre', 'dumas', 'tome', 'i', '_', 'table', 'des', 'matières', 'alexandre', 'dumas', 'i', 'marseille', "__l'arrivée", 'ii', 'le', 'père', 'et', 'le', 'fils', 'iii', 'les', 'catalans', 'iv', 'complot', 'v', 'le', 'repas', 'des', 'fiançailles', 'vi', 'le', 'substitut', 'du', 'procureur', 'du', 'roi', 'vii', "l'interrogatoire", 'viii', 'le', 'château', "d'if", 'ix', 'le', 'soir', 'des']


## b)  Calculez et affichez :
- Le nombre total de tokens
- Le nombre de tokens uniques (vocabulaire)
- La richesse lexicale (ratio unique/total)


In [84]:

print(f"Total de tokens avec NLTK : {len(tokens):,} ")

print(f"Le nombre de token unique : {len(set(tokens)):,}")

print(f'Richesse lexicale : {len(set(tokens))/len(tokens):.4f}')




Total de tokens avec NLTK : 122,202 
Le nombre de token unique : 12,645
Richesse lexicale : 0.1035


## c)  Comparez NLTK et SpaCy sur le premier chapitre (5000 premiers caractères).


### Tokenisation avec SpaCy

In [78]:
extrain_test = roman_clean[:5000]

nlp = spacy.load('fr_core_news_sm')

doc = nlp(extrain_test)

token_spacy = [token.text for token in doc]

print(f"Le nombre total de token: {len(token_spacy):,}")
print(f"Les 50 premiers tokens : {token_spacy[:50]}")

Le nombre total de token: 934
Les 50 premiers tokens : ['le', 'comte', 'de', 'monte_cristo', 'alexandre', 'dumas', 'tome', 'i', '_', 'table', 'des', 'matières', 'alexandre', 'dumas', 'i', 'marseille', '_', '_', "l'", 'arrivée', 'ii', 'le', 'père', 'et', 'le', 'fils', 'iii', 'les', 'catalans', 'iv', 'complot', 'v', 'le', 'repas', 'des', 'fiançailles', 'vi', 'le', 'substitut', 'du', 'procureur', 'du', 'roi', 'vii', "l'", 'interrogatoire', 'viii', 'le', 'château', "d'"]


## c)  Comparez NLTK et SpaCy sur le premier chapitre (5000 premiers caractères).


In [82]:
token_NLTK = word_tokenize(extrain_test)

print(f"Le nombre total de token avec NLTK sur les 5000 premiers caractères:\n{len(token_NLTK):,} tokens\n")
print(f"Le nombre total de token avec SPACY sur les 5000 premiers caractères:\n{len(token_spacy):,} tokens")



Le nombre total de token avec NLTK sur les 5000 premiers caractères:
857 tokens

Le nombre total de token avec SPACY sur les 5000 premiers caractères:
934 tokens


In [86]:

tokens_nltk = word_tokenize(extrain_test, language='french')
tokens_spacy = [token.text for token in doc if not token.is_space]

# Comparer les 30 premiers
print("NLTK vs SpaCy (30 premiers tokens) :\n")
print(f"{'NLTK':20} | {'SpaCy':20}")
print("-" * 42)
for i in range(min(30, max(len(tokens_nltk), len(tokens_spacy)))):
    nltk_t = tokens_nltk[i] if i < len(tokens_nltk) else "—"
    spacy_t = tokens_spacy[i] if i < len(tokens_spacy) else "—"
    print(f"{nltk_t:20} | {spacy_t:20}")

NLTK vs SpaCy (30 premiers tokens) :

NLTK                 | SpaCy               
------------------------------------------
le                   | le                  
comte                | comte               
de                   | de                  
monte_cristo         | monte_cristo        
alexandre            | alexandre           
dumas                | dumas               
tome                 | tome                
i                    | i                   
_                    | _                   
table                | table               
des                  | des                 
matières             | matières            
alexandre            | alexandre           
dumas                | dumas               
i                    | i                   
marseille            | marseille           
__l'arrivée          | _                   
ii                   | _                   
le                   | l'                  
père                 | arrivée         

### Notez les différences dans un tableau.

- Avec spacy plus de tokens(plus précis dans la découpe)
- Avec NLTK moins de tokens(moins précis dans la découpe)
- Spacy est plus lent 
- NLTK est plus rapide

## d)  Extrayez les 20 bigrammes les plus fréquents du roman.
Y a-t-il des noms de personnages ou des expressions caractéristiques ?
Hint : Counter(bigrams(tokens)).most_common(20)
